In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
USE_KAGGLEHUB = True

In [ ]:
import os, json
dnepozitek_polyvore_outfits_path = kagglehub.dataset_download('dnepozitek/polyvore-outfits')
dataset_root = os.path.join(dnepozitek_polyvore_outfits_path, "polyvore_outfits")

In [ ]:
mynrawu_clo_mapping_path = kagglehub.dataset_download('mynrawu/clo-mapping')
mynrawu_fitbclo_path = kagglehub.dataset_download('mynrawu/fitbclo')
google_gemma_3_transformers_gemma_3_4b_it_1_path = kagglehub.model_download('google/gemma-3/Transformers/gemma-3-4b-it/1')
print('Data source import complete.')

In [ ]:
!pip install -U transformers
!pip install datasets

In [ ]:
# === 輸出資料夾：寫進 Google Drive ===
OUT_DIR = "/content/drive/MyDrive"
os.makedirs(OUT_DIR, exist_ok=True)

# === Polyvore 資料集路徑（沿用你剛設好的 dataset_root）===
meta_path = os.path.join(dataset_root, "polyvore_item_metadata.json")
IMAGES_DIR = os.path.join(dataset_root, "images")
SPLIT_DIR  = os.path.join(dataset_root, "disjoint")

# === CLO 對照表 ===
ENSEMBLE_CLO_JSON = os.path.join(mynrawu_clo_mapping_path, "ensemble_clo.json")
SINGLE_CLO_JSON   = os.path.join(mynrawu_clo_mapping_path, "single_items_clo.json")

# === Gemma-3 模型（Transformers 轉換後目錄）===
GEMMA_MODEL_DIR = google_gemma_3_transformers_gemma_3_4b_it_1_path  # 這個路徑就是 model_download 回傳的資料夾

# === 之前的結果（可選）===
SRC_PREV = os.path.join(mynrawu_fitbclo_path, "valid_clo_results.jsonl")  # 若不存在會自動跳過

# === 本次輸出檔（寫進 Drive）===
OUT_JSON = os.path.join(OUT_DIR, "valid_clo_results.jsonl")

In [ ]:
# -*- coding: utf-8 -*-
import os, json, math, glob, re, shutil
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
# ========== 基本匯入 ==========
import pandas as pd
import torch
from transformers import pipeline
from PIL import Image  # 避免 Kaggle lazy import 問題

# Import variables from previous cells
# You might need to run the previous cells first for these variables to be defined
# from .ipynb_checkpoints.x1zPd2KJSHh3-checkpoint import OUT_DIR, GEMMA_MODEL_DIR, ENSEMBLE_CLO_JSON, SINGLE_CLO_JSON, meta_path, IMAGES_DIR, SPLIT_DIR
# from .ipynb_checkpoints.oW-X-Uz_QU0g-checkpoint import dataset_root
# from .ipynb_checkpoints.QxdJNKFfQY7T-checkpoint import mynrawu_clo_mapping_path, mynrawu_fitbclo_path, google_gemma_3_transformers_gemma_3_4b_it_1_path

# --- Define variables if not already defined (fallback) ---
# These should ideally be defined by running previous cells
if 'OUT_DIR' not in globals():
    OUT_DIR = "/content/drive/MyDrive"
if 'GEMMA_MODEL_DIR' not in globals():
    # Assuming this variable is defined in a previous cell after model_download
    # Replace with the actual path if you know it or run the cell defining it
    GEMMA_MODEL_DIR = google_gemma_3_transformers_gemma_3_4b_it_1_path # Placeholder, ensure this is correctly defined upstream

if 'ENSEMBLE_CLO_JSON' not in globals():
    ENSEMBLE_CLO_JSON = os.path.join(mynrawu_clo_mapping_path, "ensemble_clo.json") # Placeholder, ensure this is correctly defined upstream

if 'SINGLE_CLO_JSON' not in globals():
     SINGLE_CLO_JSON = os.path.join(mynrawu_clo_mapping_path, "single_items_clo.json") # Placeholder, ensure this is correctly defined upstream

if 'meta_path' not in globals():
     meta_path = os.path.join(dataset_root, "polyvore_item_metadata.json") # Placeholder, ensure this is correctly defined upstream

if 'IMAGES_DIR' not in globals():
    IMAGES_DIR = os.path.join(dataset_root, "images") # Placeholder, ensure this is correctly defined upstream

if 'SPLIT_DIR' not in globals():
     SPLIT_DIR  = os.path.join(dataset_root, "disjoint") # Placeholder, ensure this is correctly defined upstream

if 'dataset_root' not in globals():
    # Assuming this variable is defined in a previous cell after dataset_download
    # Replace with the actual path if you know it or run the cell defining it
    dnepozitek_polyvore_outfits_path = kagglehub.dataset_download('dnepozitek/polyvore-outfits') # Placeholder
    dataset_root = os.path.join(dnepozitek_polyvore_outfits_path, "polyvore_outfits") # Placeholder

if 'mynrawu_clo_mapping_path' not in globals():
    mynrawu_clo_mapping_path = kagglehub.dataset_download('mynrawu/clo-mapping') # Placeholder

if 'mynrawu_fitbclo_path' not in globals():
     mynrawu_fitbclo_path = kagglehub.dataset_download('mynrawu/fitbclo') # Placeholder

if 'google_gemma_3_transformers_gemma_3_4b_it_1_path' not in globals():
    google_gemma_3_transformers_gemma_3_4b_it_1_path = kagglehub.model_download('google/gemma-3/Transformers/gemma-3-4b-it/1') # Placeholder


MAKE_COLLAGE = True          # 批次建議關閉，節省時間/流量
# ===== Gemma-3 本地推論（多模態） =====
pipe = pipeline(
    task="image-text-to-text",
    model=GEMMA_MODEL_DIR,
    device="cuda",
    dtype=torch.bfloat16
)
# ========== 1. 載入 CLO 資料表 ==========
with open(ENSEMBLE_CLO_JSON, "r", encoding="utf-8") as f:
    ensemble_clo = json.load(f)
with open(SINGLE_CLO_JSON, "r", encoding="utf-8") as f:
    single_items_clo = json.load(f)

ensemble_clo_text = json.dumps(ensemble_clo, ensure_ascii=False)
single_items_clo_text = json.dumps(single_items_clo, ensure_ascii=False)

print("正在載入 Metadata / 設定資料集路徑...")
meta_df = pd.read_json(meta_path, orient="index")
meta_df.index = meta_df.index.astype(str)
meta_df.index.name = "item_id"

# ========== 工具函式 ==========
def image_path(item_id: str) -> Optional[str]:
    """找圖片路徑"""
    if item_id in meta_df.index:
        row = meta_df.loc[item_id]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        for key in ("image", "image_path", "filename", "img", "img_path", "img_filename"):
            v = row.get(key, None)
            if isinstance(v, str) and v.strip():
                p = os.path.join(IMAGES_DIR, v.lstrip("/")) if not os.path.isabs(v) else v
                if os.path.exists(p):
                    return p
    for ext in ("jpg","jpeg","png","JPG","JPEG","PNG"):
        p = os.path.join(IMAGES_DIR, f"{item_id}.{ext}")
        if os.path.exists(p): return p
    hits = glob.glob(os.path.join(IMAGES_DIR, "**", f"{item_id}.*"), recursive=True)
    return hits[0] if hits else None

def make_collage(image_paths: List[str], out_path: str, tile=(224,224), cols=4):
    if not image_paths: return None

    w,h = tile; rows = math.ceil(len(image_paths)/cols)
    canvas = Image.new("RGB", (cols*w, rows*h), (255,255,255))
    r=c=0
    for p in image_paths:
        try:
            im = Image.open(p).convert("RGB").resize(tile, Image.LANCZOS)
            canvas.paste(im, (c*w, r*h))
            c+=1
            if c==cols: c=0; r+=1
        except Exception as e:
            print("[WARN] 讀圖失敗：", p, e)
    canvas.save(out_path); return out_path

def _infer_from_text(row: pd.Series) -> str:
    tokens = []
    cats = row.get("categories", None)
    if isinstance(cats, list): tokens += [str(x).lower() for x in cats]
    elif isinstance(cats, str): tokens.append(cats.lower())
    title = ""
    if isinstance(row, pd.Series) and "title" in row and isinstance(row["title"], str):
        title = row["title"].lower()
    text = " ".join(tokens + [title])
    rules = [
        ("all-body", ["dress", "jumpsuit", "gown", "romper"]),
        ("outerwear", ["jacket", "coat", "trench", "parka", "cardigan", "blazer", "windbreaker", "down"]),
        ("bottoms", ["pants", "trouser", "jeans", "shorts", "skirt", "culotte", "leggings"]),
        ("tops", ["top", "t-shirt", "tee", "shirt", "blouse", "sweater", "pullover", "hoodie", "tank", "camisole"]),
        ("shoes", ["shoe", "sandal", "sneaker", "boot", "heel", "pump", "loafer", "flat", "oxford"]),
        ("bags", ["bag", "handbag", "backpack", "tote", "clutch", "purse", "satchel", "duffel"]),
        ("jewellery", ["necklace", "earring", "bracelet", "ring", "jewelry", "jewellery"]),
        ("hats", ["hat", "cap", "beanie", "beret", "fedora"]),
        ("accessories", ["belt", "scarf", "glove", "sunglass", "sunglasses", "watch", "wallet", "shawl", "tie"])
    ]
    for cat, kws in rules:
        if any(k in text for k in kws):
            return cat
    return "unknown"

def item_info(item_id: str) -> dict:
    if item_id not in meta_df.index:
        return {"id": item_id, "title": "(missing)", "category": "unknown"}
    row = meta_df.loc[item_id]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    for key in ["title", "description", "url_name", "name"]:
        val = row.get(key, None)
        if isinstance(val, str) and val.strip():
            title = val.strip()
            break
    else:
        title = "(no title)"
    cat = row.get("semantic_category", None)
    if not isinstance(cat, str) or not cat.strip():
        cat = _infer_from_text(row)
    return {"id": item_id, "title": title, "category": cat}

# ---- 輔助函式：解析模型回傳 ----
def extract_json_block(s: str):
    if not isinstance(s, str) or not s.strip():
        return None
    # 優先抓 <json>...</json>
    m = re.findall(r"<json>\s*(\{.*?\})\s*</json>", s, flags=re.DOTALL)
    candidates = m if m else []

    # 退而求其次：用大括號平衡法抓最後一個
    if not candidates:
        spans, stack = [], []
        for i, ch in enumerate(s):
            if ch == '{':
                stack.append(i)
            elif ch == '}' and stack:
                start = stack.pop()
                if not stack:
                    spans.append((start, i+1))
        candidates = [s[a:b] for (a,b) in spans]

    for raw in reversed(candidates):
        try:
            clean_raw = raw.strip().replace("“", '"').replace("”", '"')

            obj = json.loads(clean_raw)
            if isinstance(obj, dict):
                return obj
        except:
            pass
    return None

def extract_reasoning(s: str) -> str:
    if not isinstance(s, str): return ""
    m = re.search(r"\[REASONING\](.*?)(?:<json>|\Z)", s, flags=re.DOTALL|re.IGNORECASE)
    if not m: return ""
    txt = re.sub(r"\s+", " ", m.group(1)).strip()
    return txt[:300]

def fallback_total_from_text(s: str) -> Optional[float]:
    if not isinstance(s, str): return None
    # 抓像 "Estimated total CLO value: 1.15" 這種
    m = re.search(r"(?i)total\s*clo(?:\s*value)?[^0-9]*([0-9]+(?:\.[0-9]+)?)", s)
    if m:
        try:
            val = float(m.group(1))
            return val
        except:
            return None
    return None

def _is_reasonable_total(x) -> bool:
    return isinstance(x, (int, float)) and math.isfinite(x) and 0.0 <= float(x) <= 3.0


# ===== 讀入 outfit 標題對照表（掃 disjoint 優先）=====
def _parse_titles_json(js) -> Dict[str, str]:
    m = {}
    if isinstance(js, dict):
        for k, v in js.items():
            if isinstance(v, str) and v.strip():
                m[str(k)] = v.strip()
            elif isinstance(v, dict):
                t = v.get("title") or v.get("name") or v.get("outfit_title") or v.get("set_title") or v.get("url_name") or v.get("description")
                if t: m[str(k)] = str(t).strip()
    elif isinstance(js, list):
        for rec in js:
            if not isinstance(rec, dict):
                continue
            gid = rec.get("set_id") or rec.get("outfit_id") or rec.get("id") or rec.get("sid") or rec.get("qid")
            title = rec.get("title") or rec.get("name") or rec.get("outfit_title") or rec.get("set_title") or rec.get("url_name") or rec.get("description")
            if gid and title:
                m[str(gid)] = str(title).strip()
    return m

def load_outfit_titles_map(base_dir: str) -> Dict[str, str]:
    candidates = [
        os.path.join(base_dir, "polyvore_outfits", "disjoint", "polyvore_outfit_titles.json"),
        os.path.join(base_dir, "polyvore_outfits", "polyvore_outfit_titles.json"),
    ]
    titles = {}
    hit_files = []
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    js = json.load(f)
                part = _parse_titles_json(js)
                if part:
                    titles.update(part)
                    hit_files.append(p)
            except Exception as e:
                print(f"[WARN] load titles {p} failed:", e)
    if not titles:
        for p in glob.glob(os.path.join(base_dir, "polyvore_outfits", "**", "*title*.json"), recursive=True):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    js = json.load(f)
                part = _parse_titles_json(js)
                if part:
                    titles.update(part)
                    hit_files.append(p)
            except:
                pass
    print(f"[INFO] outfit titles loaded: {len(titles)} entries from {len(hit_files)} file(s).")
    return titles

OUTFIT_TITLES = load_outfit_titles_map(os.path.dirname(dataset_root))

# ========== 2. 讀 disjoint/valid.json ==========
def _load_disjoint_valid() -> List[dict]:
    candidates = [
        os.path.join(SPLIT_DIR, "valid.json"),
        os.path.join(dataset_root, "polyvore_outfits", "valid.json"),
    ]
    for p in candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                js = json.load(f)
            if isinstance(js, list):
                return js
            if isinstance(js, dict):
                for k in ("valid", "items", "outfits", "data", "records"):
                    if k in js and isinstance(js[k], list):
                        return js[k]
            raise ValueError(f"無法解析 valid.json 結構：{p}")
    raise FileNotFoundError("找不到 disjoint/valid.json，請確認資料夾路徑。")

def _extract_outfit(rec: dict) -> Tuple[str, List[str]]:
    """回傳 (outfit_id, item_ids)；盡量相容不同欄位命名"""
    gid = str(rec.get("set_id") or rec.get("outfit_id") or rec.get("id") or rec.get("sid") or rec.get("qid") or rec.get("uid") or "")
    items = rec.get("items") or rec.get("products") or rec.get("images") or rec.get("members") or []
    item_ids = []
    if isinstance(items, list):
        for it in items:
            if isinstance(it, dict):
                iid = it.get("item_id") or it.get("id") or it.get("product_id") or it.get("member_id")
            else:
                iid = it
            if iid is not None:
                item_ids.append(str(iid))
    return gid, item_ids

valid_records = _load_disjoint_valid()
print(f"[INFO] disjoint valid 載入 {len(valid_records)} 筆")

# ---- JSONL I/O ----
def read_jsonl(path: str) -> list:
    if not os.path.exists(path):
        return []
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

def append_jsonl(path: str, rec: dict):
    line = json.dumps(rec, ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line); f.flush(); os.fsync(f.fileno())

SYSTEM_PROMPT = """You are a clothing thermal comfort expert.
    Your task is to estimate the total CLO insulation value of an outfit.

    Return exactly two blocks:
    [REASONING]
    - <=4 bullets, <=280 chars, no tables, no lists from data.
    - Briefly note included layers and ignored accessories.

    <json>{"total_clo": number}</json>
    Only one JSON object. Do not echo inputs.
    """


# ========== 3. 單筆處理（估 CLO，不用 FITB） ==========
def process_one_outfit(sample_idx: int, rec: dict) -> Optional[float]:
    gid, item_ids = _extract_outfit(rec)
    if not item_ids:
        print(f"[WARN] outfit {gid or sample_idx} 無 item_ids，跳過")
        return None

    items_info = [item_info(i) for i in item_ids]
    items_block = "\n".join([f"- id={it['id']} | category={it['category']} | desc={it['title']}" for it in items_info])

    outfit_title = (OUTFIT_TITLES.get(gid) or rec.get("title") or rec.get("name") or rec.get("set_title") or rec.get("description"))
    outfit_desc = outfit_title.strip() if isinstance(outfit_title, str) and outfit_title.strip() \
                  else "Outfit with: " + ", ".join(sorted(set(it["category"] for it in items_info)))
    #print("=== 單品資訊 (items_info) ===")
    #print(items_block)
    #print("\n=== 套裝描述 (outfit_desc) ===")
    #print(outfit_desc)
    ALL = None
    # 在 process_one_outfit() 內，建立拼貼輸出資料夾於 Drive
    if MAKE_COLLAGE:
        imgs = [image_path(i) for i in item_ids if i]
        imgs = [p for p in imgs if p]
        if imgs:
            coll_dir = os.path.join(OUT_DIR, "collages_valid_demo")
            os.makedirs(coll_dir, exist_ok=True)
            ALL = make_collage(imgs, os.path.join(coll_dir, f"{gid or sample_idx}.jpg"), cols=4)


    # ---- Prompt ----
    USER_PROMPT = (
    f"""
    Outfit description: {outfit_desc}

    Items:
    {items_block}

    Reference CLO knowledge:
    Ensemble CLO values:
    {ensemble_clo_text}

    Single item CLO values:
    {single_items_clo_text}

    Now, estimate the total CLO value of this outfit.
    Return two blocks exactly:
    [REASONING] <=4 bullets, <=280 chars.
    <json>"""
        + '{"total_clo": number}'
        + """</json>
    """
    )


    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": []}
    ]
    if ALL:
        messages[1]["content"].append({"type": "image","image": ALL})
    messages[1]["content"].append({"type": "text", "text": USER_PROMPT})

    raw_text = ""
    try:
        output = pipe(text=messages, max_new_tokens=500)
        raw_text = output[0]["generated_text"][-1]["content"]
    except Exception as e:
        print(f"[ERROR] 推論失敗：{e}")
        #append_jsonl(OUT_JSON, { "set_id": gid, "status": "failed", })
        return None

    parsed = extract_json_block(raw_text)
    total_clo = parsed.get("total_clo") if isinstance(parsed, dict) else None

    if not _is_reasonable_total(total_clo):
        cand = fallback_total_from_text(raw_text)
        if _is_reasonable_total(cand):
            total_clo = cand

    rec_out = {
        "set_id": gid,
        "total_clo": total_clo
    }

    if isinstance(total_clo, (int, float)) and math.isfinite(total_clo):
        append_jsonl(OUT_JSON, rec_out)
        return total_clo
    else:
        #append_jsonl(OUT_JSON, { "set_id": gid, "status": "failed", "raw_output": raw_text })
        print(raw_text)
        return None

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import math
import shutil # Import shutil for file operations
from tqdm import tqdm # Import tqdm for progress bar

# Assuming these variables are defined in previous cells
# If not, you might need to run those cells first or define them here as a fallback
# from .ipynb_checkpoints.x1zPd2KJSHh3-checkpoint import OUT_DIR, SRC_PREV, valid_records # Import valid_records
# from .ipynb_checkpoints._nBd0z8-Qzqu-checkpoint import process_one_outfit # Import process_one_outfit

# --- Define variables if not already defined (fallback) ---
if 'OUT_DIR' not in globals():
    OUT_DIR = "/content/drive/MyDrive" # Placeholder

if 'SRC_PREV' not in globals():
    SRC_PREV = os.path.join(OUT_DIR, "valid_clo_results.jsonl") # Placeholder, assuming default output name

if 'valid_records' not in globals():
    # This should be loaded from the valid.json file as in the previous cell
    # You might need to run the cell that loads valid_records first
    print("[WARN] 'valid_records' not found. Attempting to load...")
    try:
        # This logic is duplicated from the previous cell, consider refactoring
        def _load_disjoint_valid(dataset_root, SPLIT_DIR) -> list:
             candidates = [
                os.path.join(SPLIT_DIR, "valid.json"),
                os.path.join(dataset_root, "polyvore_outfits", "valid.json"),
            ]
             for p in candidates:
                if os.path.exists(p):
                    with open(p, "r", encoding="utf-8") as f:
                        js = json.load(f)
                    if isinstance(js, list):
                        return js
                    if isinstance(js, dict):
                        for k in ("valid", "items", "outfits", "data", "records"):
                            if k in js and isinstance(js[k], list):
                                return js[k]
                    raise ValueError(f"無法解析 valid.json 結構：{p}")
             raise FileNotFoundError("找不到 disjoint/valid.json，請確認資料夾路徑。")

        # Attempt to use dataset_root and SPLIT_DIR if they exist, otherwise use placeholders
        dataset_root_fallback = globals().get('dataset_root', '/content') # Use a default if not defined
        SPLIT_DIR_fallback = globals().get('SPLIT_DIR', '/content/polyvore_outfits/disjoint') # Use a default if not defined

        valid_records = _load_disjoint_valid(dataset_root_fallback, SPLIT_DIR_fallback)
        print(f"[INFO] Fallback loaded {len(valid_records)} records for 'valid_records'.")
    except Exception as e:
        print(f"[ERROR] Failed to load 'valid_records' in fallback: {e}")
        valid_records = [] # Set to empty list if loading fails


# Assuming process_one_outfit is defined in a previous cell.
# If not, you need to run that cell or include its definition here.
if 'process_one_outfit' not in globals():
    print("[ERROR] 'process_one_outfit' function not found. Please ensure the cell defining it is run.")
    # You might need to copy the entire function definition here as a fallback
    # or provide a simplified placeholder function that indicates the error.
    def process_one_outfit(sample_idx, rec):
        print(f"[ERROR] process_one_outfit called but not defined. Outfit ID: {rec.get('set_id', 'N/A')}")
        return None



OUT_JSON = os.path.join(OUT_DIR, "valid_clo_results.jsonl")


# ========== 4. 主流程==========
RUN_NUM = None
def is_finite_num(x):
    return isinstance(x, (int, float)) and math.isfinite(x)

# 1) 若有舊成果就複製到輸出
if os.path.exists(SRC_PREV) and not os.path.exists(OUT_JSON):
    shutil.copy(SRC_PREV, OUT_JSON)
    print(f"[INFO] 複製既有檔案：{SRC_PREV} -> {OUT_JSON}")
elif not os.path.exists(OUT_JSON):
    # 沒舊檔就新建空檔
    open(OUT_JSON, "w", encoding="utf-8").close()
    print(f"[INFO] 建立新檔：{OUT_JSON}")

# 2) 讀取已完成或失敗的索引
completed_indices = set()
if os.path.exists(OUT_JSON):
    with open(OUT_JSON, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                # 成功的（有 total_clo）
                if isinstance(data, dict) and "set_id" in data:
                    if is_finite_num(data.get("total_clo")):
                        completed_indices.add(data["set_id"])
                    # 失敗的也略過
                    elif data.get("status") == "failed":
                        completed_indices.add(data["set_id"])
            except:
                pass


def main():
    # n：本輪想新完成的數目；None = 把剩下的都跑完
    n = RUN_NUM

    # 已完成（檔案裡的成功樣本）數量
    already_done = len(completed_indices)

    # 還沒完成的索引（用 set_id 比對）
    remaining_indices = [i for i in range(len(valid_records))
                        if valid_records[i]["set_id"] not in completed_indices]

    # 本輪目標要新完成幾筆
    target_new = len(remaining_indices) if n is None else min(n, len(remaining_indices))

    print(f"[INFO] 已完成: {already_done}，本輪目標新增: {target_new}，資料總筆數: {len(valid_records)}")

    success_new = 0

    with tqdm(total=already_done + target_new, initial=already_done, desc="Processing disjoint valid (CLO)") as pbar:
        for i in remaining_indices:
            if n is not None and success_new >= n:
                break
            try:
                clo = process_one_outfit(i, valid_records[i])
                # 只有成功才前進進度條
                if is_finite_num(clo):
                    success_new += 1
                    pbar.update(1)
            except Exception as e:
                print(f"[WARN] outfit idx={i} failed: {e}")

    print(f"\n[FINAL] 本輪成功 {success_new}/{target_new}，累計完成 {already_done + success_new}/{len(valid_records)}")

if __name__ == "__main__":
    main()

# 檢查重複元素

In [ ]:
import json
import os

# --- 設定 ---
# 確認這個路徑是您含有重複資料的結果檔
RESULTS_JSONL_TO_CLEAN = "/content/drive/MyDrive/valid_clo_results.jsonl" # Changed from train_clo_results.jsonl
# ---

def deduplicate_results(filepath: str):
    """讀取 JSONL 檔案，移除重複的 set_id，並覆寫回原檔案。"""
    if not os.path.exists(filepath):
        print(f"錯誤：找不到檔案 {filepath}")
        return

    print(f"--- 開始清理檔案: {filepath} ---")

    # 使用字典來儲存每個 set_id 的最後一筆紀錄
    # 這樣可以確保我們保留的是最新版本
    final_records = {}

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                # 確保紀錄中有 set_id
                if "set_id" in record:
                    final_records[record["set_id"]] = record
            except json.JSONDecodeError:
                print(f"警告：跳過一行無法解析的內容: {line.strip()}")

    original_count = sum(1 for line in open(filepath, 'r'))
    deduplicated_count = len(final_records)

    if original_count == deduplicated_count:
        print("檔案中沒有發現重複的 set_id，無需清理。")
        return

    print(f"原始筆數: {original_count}")
    print(f"去重複後筆數: {deduplicated_count}")

    # 建立備份
    backup_path = filepath + ".bak"
    print(f"建立備份檔案: {backup_path}")
    os.rename(filepath, backup_path)

    # 將清理後的唯一紀錄寫回原檔案
    # 我們從備份檔案中讀取原始順序，盡量保持檔案結構
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            seen_ids = set()
            with open(backup_path, "r", encoding="utf-8") as bf:
                for line in bf:
                    try:
                        rec = json.loads(line)
                        set_id = rec.get("set_id")
                        if set_id and set_id not in seen_ids:
                             # 從 final_records 中寫入該 id 的最終版本
                            f.write(json.dumps(final_records[set_id], ensure_ascii=False) + "\n")
                            seen_ids.add(set_id)
                    except:
                        continue # 忽略備份檔中的錯誤行
        print(f"成功將 {deduplicated_count} 筆唯一紀錄寫回 {filepath}")
    except Exception as e:
        print(f"錯誤：寫回檔案失敗 - {e}")
        print(f"您的原始檔案已備份至 {backup_path}")


# 執行清理
if __name__ == "__main__":
    deduplicate_results(RESULTS_JSONL_TO_CLEAN)

# 處理進度

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import math

# Assuming these variables are defined in previous cells
# If not, you might need to run those cells first or define them here as a fallback
# from .ipynb_checkpoints.x1zPd2KJSHh3-checkpoint import SPLIT_DIR, OUT_JSON

# --- Define variables if not already defined (fallback) ---
if 'SPLIT_DIR' not in globals():
     SPLIT_DIR  = os.path.join(globals().get('dataset_root', '/content/polyvore_outfits'), "disjoint") # Placeholder

if 'OUT_JSON' not in globals():
    OUT_JSON = os.path.join(globals().get('OUT_DIR', '/content/drive/MyDrive'), "valid_clo_results.jsonl") # Placeholder


def is_finite_num(x):
    """檢查一個值是否為有效的有限數字（整數或浮點數）"""
    return isinstance(x, (int, float)) and math.isfinite(x)

def load_total_records(split_dir: str) -> int:
    """載入驗證資料以確定總共有多少筆 outfit 需要處理"""
    valid_json_path = os.path.join(split_dir, "valid.json") # Changed from train.json
    if not os.path.exists(valid_json_path):
        print(f"錯誤：在 '{valid_json_path}' 找不到驗證資料檔") # Changed from訓練資料檔
        print("請確認此腳本頂部的 SPLIT_DIR 變數設定是否正確。")
        return 0
    try:
        with open(valid_json_path, "r", encoding="utf-8") as f: # Changed from train_json_path
            valid_data = json.load(f) # Changed from train_data
        if isinstance(valid_data, list): # Changed from train_data
            return len(valid_data) # Changed from train_data
        else:
            print(f"警告：'{valid_json_path}' 的格式非預期，無法計算總數。") # Changed from train_json_path
            return 0
    except Exception as e:
        print(f"讀取驗證資料時發生錯誤：{e}") # Changed from 訓練資料
        return 0

def count_completed_items(output_file: str) -> int:
    """計算輸出 JSONL 檔案中已成功處理的項目數量"""
    if not os.path.exists(output_file):
        print(f"提示：輸出檔 '{output_file}' 不存在，假設已完成 0 項。")
        return 0

    completed_ids = set()
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                    # 如果紀錄包含 'set_id' 且 'total_clo' 為有效數字，則視為成功
                    if isinstance(data, dict) and "set_id" in data and is_finite_num(data.get("total_clo")):
                        completed_ids.add(data["set_id"])
                except json.JSONDecodeError:
                    # 忽略損毀的行
                    continue
        return len(completed_ids)
    except Exception as e:
        print(f"讀取輸出檔 '{output_file}' 時發生錯誤：{e}")
        return 0

def main():
    """主函式，用於檢查並顯示處理進度"""
    print("--- 正在檢查處理進度 ---")

    # 直接使用在檔案頂部設定的變數
    total_items = load_total_records(SPLIT_DIR)
    if total_items == 0:
        print("無法確定項目總數，程式結束。")
        return

    completed_items = count_completed_items(OUT_JSON)

    print(f"\n輸出檔: '{OUT_JSON}'")
    print(f"資料集檔案: '{os.path.join(SPLIT_DIR, 'valid.json')}'") # Changed from train.json

    if total_items > 0:
        percentage = (completed_items / total_items) * 100
        print("\n--- 進度總結 ---")
        print(f"已完成: {completed_items} / {total_items} ({percentage:.2f}%)")
        print("----------------")
    else:
        print("\n無法計算進度百分比。")

if __name__ == "__main__":
    main()

# 異常值修正

In [ ]:
import os # Import os
import json # Import json
import math # Import math
import shutil # Import shutil
from tqdm import tqdm # Import tqdm
import pandas as pd # Import pandas

# Assuming these variables are defined in previous cells
# If not, you might need to run those cells first or define them here as a fallback
# from .ipynb_checkpoints.x1zPd2KJSHh3-checkpoint import OUT_JSON, SPLIT_DIR, valid_records # Import valid_records
# from .ipynb_checkpoints._nBd0z8-Qzqu-checkpoint import read_jsonl, process_one_outfit # Import read_jsonl and process_one_outfit

# --- Define variables if not already defined (fallback) ---
if 'OUT_JSON' not in globals():
    OUT_JSON = os.path.join(globals().get('OUT_DIR', '/content/drive/MyDrive'), "valid_clo_results.jsonl") # Placeholder

if 'SPLIT_DIR' not in globals():
    SPLIT_DIR  = os.path.join(globals().get('dataset_root', '/content/polyvore_outfits'), "disjoint") # Placeholder

if 'valid_records' not in globals():
    # This should be loaded from the valid.json file as in the previous cell
    # You might need to run the cell that loads valid_records first
    print("[WARN] 'valid_records' not found. Attempting to load...")
    try:
        # This logic is duplicated from the previous cell, consider refactoring
        def _load_disjoint_valid(dataset_root, SPLIT_DIR) -> list:
             candidates = [
                os.path.join(SPLIT_DIR, "valid.json"),
                os.path.join(dataset_root, "polyvore_outfits", "valid.json"),
            ]
             for p in candidates:
                if os.path.exists(p):
                    with open(p, "r", encoding="utf-8") as f:
                        js = json.load(f)
                    if isinstance(js, list):
                        return js
                    if isinstance(js, dict):
                        for k in ("valid", "items", "outfits", "data", "records"):
                            if k in js and isinstance(js[k], list):
                                return js[k]
                    raise ValueError(f"無法解析 valid.json 結構：{p}")
             raise FileNotFoundError("找不到 disjoint/valid.json，請確認資料夾路徑。")

        # Attempt to use dataset_root and SPLIT_DIR if they exist, otherwise use placeholders
        dataset_root_fallback = globals().get('dataset_root', '/content') # Use a default if not defined
        SPLIT_DIR_fallback = globals().get('SPLIT_DIR', '/content/polyvore_outfits/disjoint') # Use a default if not defined

        valid_records = _load_disjoint_valid(dataset_root_fallback, SPLIT_DIR_fallback)
        print(f"[INFO] Fallback loaded {len(valid_records)} records for 'valid_records'.")
    except Exception as e:
        print(f"[ERROR] Failed to load 'valid_records' in fallback: {e}")
        valid_records = [] # Set to empty list if loading fails


# Assuming read_jsonl and process_one_outfit are defined in a previous cell.
# If not, you need to run that cell or include their definitions here.
if 'read_jsonl' not in globals():
     print("[ERROR] 'read_jsonl' function not found. Please ensure the cell defining it is run.")
     # You might need to copy the entire function definition here as a fallback
     def read_jsonl(path: str) -> list:
         if not os.path.exists(path):
             return []
         data = []
         with open(path, "r", encoding="utf-8") as f:
             for line in f:
                 line = line.strip()
                 if not line: continue
                 try:
                     data.append(json.loads(line))
                 except:
                     pass
         return data
     print("[INFO] Added fallback definition for 'read_jsonl'.")


if 'process_one_outfit' not in globals():
    print("[ERROR] 'process_one_outfit' function not found. Please ensure the cell defining it is run.")
    # You might need to copy the entire function definition here as a fallback
    # or provide a simplified placeholder function that indicates the error.
    def process_one_outfit(sample_idx, rec):
        print(f"[ERROR] process_one_outfit called but not defined. Outfit ID: {rec.get('set_id', 'N/A')}")
        return None


def analyze_and_correct_outliers(results_path: str):
    """
    分析 CLO 結果檔案，找出並自動修正異常值。
    """
    print(f"--- 開始分析與修正檔案: {results_path} ---")
    if not os.path.exists(results_path):
        print("錯誤：找不到結果檔。")
        return

    # 1. 載入所有結果並找出異常值
    all_results = read_jsonl(results_path)
    if not all_results:
        print("結果檔是空的，無法進行分析。")
        return

    df = pd.DataFrame(all_results)
    # 確保 'total_clo' 欄位存在且為數值型態
    if "total_clo" not in df.columns:
        print("錯誤：結果檔中找不到 'total_clo' 欄位。")
        return

    df["total_clo"] = pd.to_numeric(df["total_clo"], errors='coerce')

    # 在計算統計數據前，先過濾掉所有不合理的數據（空值 和 小於0的值）
    print(f"原始數據筆數: {len(df)}")
    cleaned_df = df.dropna(subset=['total_clo'])
    cleaned_df = cleaned_df[cleaned_df['total_clo'] >= 0]
    print(f"清理後，用於統計的有效數據筆數: {len(cleaned_df)}")

    # 使用清理後的乾淨數據進行統計計算
    clo_values = cleaned_df["total_clo"]

    q1, q3 = clo_values.quantile(0.25), clo_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # 即使數據清理過，我們仍然要確保計算出的下限不為負
    lower_bound = max(0, lower_bound)

    outliers_df = df[(df["total_clo"] < lower_bound) | (df["total_clo"] > upper_bound)]

    if outliers_df.empty:
        print("\n恭喜！在定義的範圍內未找到任何異常值。")
        return

    print(f"找到 {len(outliers_df)} 個異常值 (正常範圍: {lower_bound:.2f} ~ {upper_bound:.2f})")

    # 建立 valid_records 的 set_id 對照表以便快速查找
    valid_records_map = {rec["set_id"]: rec for rec in valid_records} # Changed from train_records_map

    # 2. 迭代處理異常值
    results_map = {res["set_id"]: res for res in all_results}
    corrected_count = 0
    failed_count = 0

    print("--- 開始重新處理異常值 ---")
    # 將 DataFrame 轉為字典列表以進行迭代
    for outlier in tqdm(outliers_df.to_dict('records'), total=len(outliers_df), desc="修正異常值"):
        set_id = outlier["set_id"]
        original_clo = outlier["total_clo"]

        # 找到對應的原始 outfit 資料
        original_record = valid_records_map.get(str(set_id)) # Changed from train_records_map
        if not original_record:
            tqdm.write(f"警告：在 valid.json 中找不到 set_id {set_id} 的原始資料，跳過。") # Changed from train.json
            continue

        new_clo = process_one_outfit(-1, original_record)

        # 3. 判斷新結果是否合理並準備更新
        if new_clo is not None and 0.0 <= new_clo <= 3.0:
            results_map[set_id]["total_clo"] = new_clo
            corrected_count += 1
            tqdm.write(f"修正成功: {set_id} (原: {original_clo:.2f} -> 新: {new_clo:.2f})")
        else:
            failed_count += 1
            tqdm.write(f"修正失敗: {set_id} (原: {original_clo:.2f}, 新結果: {new_clo or '無效'})")

    # 4. 寫回修正後的結果
    if corrected_count > 0:
        backup_path = results_path + ".bak"
        print(f"\n--- 完成處理，正在寫回 {corrected_count} 筆修正 ---")
        print(f"建立備份檔案: {backup_path}")
        shutil.copy(results_path, backup_path)

        # 保持原始順序寫回
        original_order_ids = [res["set_id"] for res in all_results]
        with open(results_path, "w", encoding="utf-8") as f:
            for set_id in original_order_ids:
                if set_id in results_map:
                    f.write(json.dumps(results_map[set_id], ensure_ascii=False) + "\n")
        print("檔案更新完成！")

    print("\n--- 修正總結 ---")
    print(f"成功修正: {corrected_count} 個")
    print(f"無法修正: {failed_count} 個")

if __name__ == "__main__":
    # 第二階段：執行新增的異常值分析與修正流程
    # 注意：我們不再 append 到舊檔案，而是直接讀取、修正並覆寫
    print("\n--- 第二階段：開始分析並修正異常值 ---")
    # 執行修正
    analyze_and_correct_outliers(OUT_JSON)
    print("\n--- 所有流程結束 ---")

# 小樣本抽查

In [7]:
# ============================================================
# Cell: 小樣本抽查（本地 WSL 版，可單獨運行）
# ============================================================
import os, json, math, random, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ================================================================
# 路徑設定
# ================================================================
BASE_DIR     = "/home/ester/valid_description"

dataset_root      = os.path.join(BASE_DIR, "polyvore_data", "polyvore_outfits")
IMAGES_DIR        = os.path.join(dataset_root, "images")
SPLIT_DIR         = os.path.join(dataset_root, "disjoint")
meta_path         = os.path.join(dataset_root, "polyvore_item_metadata.json")
ENSEMBLE_CLO_JSON = os.path.join(BASE_DIR, "prompt_CLO", "ensemble_clo.json")
SINGLE_CLO_JSON   = os.path.join(BASE_DIR, "prompt_CLO", "single_items_clo.json")
OUT_JSON          = os.path.join(BASE_DIR, "generate_CLO", "valid_clo_results.jsonl")

# ---- 路徑檢查 ----
print("=== 路徑檢查 ===")
for label, p in [
    ("dataset_root",      dataset_root),
    ("IMAGES_DIR",        IMAGES_DIR),
    ("SPLIT_DIR",         SPLIT_DIR),
    ("meta_path",         meta_path),
    ("ENSEMBLE_CLO_JSON", ENSEMBLE_CLO_JSON),
    ("SINGLE_CLO_JSON",   SINGLE_CLO_JSON),
    ("OUT_JSON",          OUT_JSON),
]:
    status = "✅" if os.path.exists(p) else "❌ 找不到"
    print(f"  {status}  {label}: {p}")
print()

# ================================================================
# 設定
# ================================================================
SAMPLE_N    = 5     # 要抽幾筆
RANDOM_SEED = 42    # None = 每次隨機

# ================================================================
# 載入資料
# ================================================================
print("[INFO] 載入 polyvore_item_metadata.json ...")
meta_df = pd.read_json(meta_path, orient="index")
meta_df.index = meta_df.index.astype(str)
meta_df.index.name = "item_id"
print(f"       meta_df shape: {meta_df.shape}")

print("[INFO] 載入 CLO mapping ...")
with open(ENSEMBLE_CLO_JSON, "r", encoding="utf-8") as f:
    ensemble_clo = json.load(f)
with open(SINGLE_CLO_JSON, "r", encoding="utf-8") as f:
    single_items_clo = json.load(f)
print("       CLO mapping 載入完成")

# ---- valid.json ----
def _load_valid_records() -> list:
    candidates = [
        os.path.join(SPLIT_DIR,    "valid.json"),
        os.path.join(dataset_root, "valid.json"),
    ]
    for p in candidates:
        if not os.path.exists(p): continue
        with open(p, "r", encoding="utf-8") as f:
            js = json.load(f)
        if isinstance(js, list): return js
        if isinstance(js, dict):
            for k in ("valid", "items", "outfits", "data", "records"):
                if k in js and isinstance(js[k], list): return js[k]
    raise FileNotFoundError("找不到 valid.json，請確認 SPLIT_DIR 路徑")

valid_records = _load_valid_records()
print(f"[INFO] valid_records：{len(valid_records)} 筆")

# ---- outfit titles ----
def _load_outfit_titles() -> dict:
    titles = {}
    candidates = [
        os.path.join(dataset_root, "disjoint", "polyvore_outfit_titles.json"),
        os.path.join(dataset_root, "polyvore_outfit_titles.json"),
    ]
    candidates += glob.glob(
        os.path.join(dataset_root, "**", "*title*.json"), recursive=True
    )
    for p in candidates:
        if not os.path.exists(p): continue
        try:
            with open(p, "r", encoding="utf-8") as f:
                js = json.load(f)
            if isinstance(js, dict):
                for k, v in js.items():
                    if isinstance(v, str) and v.strip():
                        titles[str(k)] = v.strip()
                    elif isinstance(v, dict):
                        t = (v.get("title") or v.get("name") or
                             v.get("outfit_title") or v.get("description"))
                        if t: titles[str(k)] = str(t).strip()
            elif isinstance(js, list):
                for rec in js:
                    if not isinstance(rec, dict): continue
                    gid = (rec.get("set_id") or rec.get("outfit_id") or rec.get("id"))
                    ttl = (rec.get("title") or rec.get("name") or
                           rec.get("outfit_title") or rec.get("description"))
                    if gid and ttl: titles[str(gid)] = str(ttl).strip()
        except Exception as e:
            print(f"  [WARN] {p}: {e}")
    print(f"[INFO] outfit titles：{len(titles)} 筆")
    return titles

OUTFIT_TITLES = _load_outfit_titles()

# ================================================================
# 工具函式
# ================================================================
def _img_path(item_id: str):
    if item_id in meta_df.index:
        row = meta_df.loc[item_id]
        if isinstance(row, pd.DataFrame): row = row.iloc[0]
        for key in ("image", "image_path", "filename", "img", "img_path", "img_filename"):
            v = row.get(key, None)
            if isinstance(v, str) and v.strip():
                p = os.path.join(IMAGES_DIR, v.lstrip("/")) if not os.path.isabs(v) else v
                if os.path.exists(p): return p
    for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
        p = os.path.join(IMAGES_DIR, f"{item_id}.{ext}")
        if os.path.exists(p): return p
    hits = glob.glob(os.path.join(IMAGES_DIR, "**", f"{item_id}.*"), recursive=True)
    return hits[0] if hits else None

def _item_title(item_id: str) -> str:
    if item_id in meta_df.index:
        row = meta_df.loc[item_id]
        if isinstance(row, pd.DataFrame): row = row.iloc[0]
        for key in ["title", "description", "url_name", "name"]:
            v = row.get(key, None)
            if isinstance(v, str) and v.strip(): return v.strip()
    return "(no title)"

def _extract_item_ids(rec: dict) -> list:
    items = (rec.get("items") or rec.get("products") or
             rec.get("images") or rec.get("members") or [])
    ids = []
    for it in items:
        iid = (it.get("item_id") or it.get("id") or
               it.get("product_id") or it.get("member_id")) if isinstance(it, dict) else it
        if iid is not None: ids.append(str(iid))
    return ids

def _flatten_clo(clo_obj, source: str) -> list:
    entries = []
    if isinstance(clo_obj, dict):
        for k, v in clo_obj.items():
            if isinstance(v, (int, float)) and math.isfinite(v):
                entries.append((str(k), float(v), source))
            elif isinstance(v, dict):
                clo_val = v.get("clo") or v.get("value") or v.get("clo_value")
                if isinstance(clo_val, (int, float)) and math.isfinite(clo_val):
                    entries.append((str(k), float(clo_val), source))
    elif isinstance(clo_obj, list):
        for item in clo_obj:
            if isinstance(item, dict):
                label = (item.get("label") or item.get("name") or
                         item.get("category") or str(item))
                val   = (item.get("clo") or item.get("value") or item.get("clo_value"))
                if isinstance(val, (int, float)) and math.isfinite(val):
                    entries.append((str(label), float(val), source))
    return entries

all_refs = (
    _flatten_clo(single_items_clo, "single_item") +
    _flatten_clo(ensemble_clo,     "ensemble")
)
print(f"[INFO] CLO 參考條目："
      f"single_item={len(_flatten_clo(single_items_clo, 'single_item'))}, "
      f"ensemble={len(_flatten_clo(ensemble_clo, 'ensemble'))}")

def find_nearest_ref(total_clo: float):
    """回傳 (label, ref_value, source)"""
    if not all_refs: return ("(no ref)", None, "N/A")
    return min(all_refs, key=lambda x: abs(x[1] - total_clo))

# ================================================================
# 讀取結果檔 & 建立 set_id → valid_record 對照
# ================================================================
results = []
if not os.path.exists(OUT_JSON):
    print(f"[ERROR] 找不到結果檔：{OUT_JSON}")
else:
    with open(OUT_JSON, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rec = json.loads(line)
                v   = rec.get("total_clo")
                if isinstance(v, (int, float)) and math.isfinite(v):
                    results.append(rec)
            except: pass
    print(f"[INFO] 有效結果筆數：{len(results)}")

set_id_to_rec = {
    str(r.get("set_id") or r.get("outfit_id") or r.get("id") or ""): r
    for r in valid_records
    if (r.get("set_id") or r.get("outfit_id") or r.get("id"))
}

# ================================================================
# 抽樣 & 顯示
# ================================================================
if not results:
    print("[ERROR] 沒有可用資料，請確認 OUT_JSON 路徑")
else:
    if RANDOM_SEED is not None: random.seed(RANDOM_SEED)
    sample = random.sample(results, min(SAMPLE_N, len(results)))

    TILE = (160, 160)
    COLS = 4

    for idx, res in enumerate(sample):
        set_id    = str(res["set_id"])
        total_clo = float(res["total_clo"])
        vr        = set_id_to_rec.get(set_id, {})
        item_ids  = _extract_item_ids(vr)

        outfit_title = (
            OUTFIT_TITLES.get(set_id)
            or vr.get("title") or vr.get("name") or vr.get("set_title")
            or "(no title)"
        )
        ref_label, ref_value, ref_source = find_nearest_ref(total_clo)

        # ---- 文字資訊 ----
        print("=" * 65)
        print(f"  [{idx+1}/{len(sample)}]  set_id       : {set_id}")
        print(f"           outfit title : {outfit_title}")
        print(f"           total_clo    : {total_clo:.4f}")
        print(f"  items ({len(item_ids)}) :")
        for iid in item_ids:
            print(f"      {iid:>12}  {_item_title(iid)}")
        print(f"  最接近參考值 : {ref_value}  "
              f"(source={ref_source}, label={ref_label})")

        # ---- 拼貼圖 ----
        imgs = [p for p in (_img_path(i) for i in item_ids if i) if p]
        if imgs:
            cols   = min(COLS, len(imgs))
            rows   = math.ceil(len(imgs) / cols)
            canvas = Image.new("RGB", (cols * TILE[0], rows * TILE[1]), (230, 230, 230))
            r = c  = 0
            for p in imgs:
                try:
                    im = Image.open(p).convert("RGB").resize(TILE, Image.LANCZOS)
                    canvas.paste(im, (c * TILE[0], r * TILE[1]))
                    c += 1
                    if c == cols: c, r = 0, r + 1
                except Exception as e:
                    print(f"  [WARN] 圖片讀取失敗: {p} — {e}")

            fig, ax = plt.subplots(figsize=(cols * 1.9, rows * 1.9 + 0.6))
            ax.imshow(np.array(canvas))
            ax.axis("off")
            ax.set_title(
                f"set_id={set_id}  total_clo={total_clo:.3f}\n"
                f"nearest ref={ref_value}  ({ref_source} / {ref_label})\n"
                f"{outfit_title[:70]}",
                fontsize=8, pad=6
            )
            plt.tight_layout()
            plt.show()
        else:
            print("  [INFO] 無法取得任何圖片")

    print("=" * 65)
    print("[完成] 小樣本抽查結束")

=== 路徑檢查 ===
  ✅  dataset_root: /home/ester/valid_description/polyvore_data/polyvore_outfits
  ✅  IMAGES_DIR: /home/ester/valid_description/polyvore_data/polyvore_outfits/images
  ✅  SPLIT_DIR: /home/ester/valid_description/polyvore_data/polyvore_outfits/disjoint
  ✅  meta_path: /home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_item_metadata.json
  ✅  ENSEMBLE_CLO_JSON: /home/ester/valid_description/prompt_CLO/ensemble_clo.json
  ✅  SINGLE_CLO_JSON: /home/ester/valid_description/prompt_CLO/single_items_clo.json
  ✅  OUT_JSON: /home/ester/valid_description/generate_CLO/valid_clo_results.jsonl

[INFO] 載入 polyvore_item_metadata.json ...
       meta_df shape: (251008, 7)
[INFO] 載入 CLO mapping ...
       CLO mapping 載入完成
[INFO] valid_records：3000 筆
[INFO] outfit titles：57528 筆
[INFO] CLO 參考條目：single_item=0, ensemble=104
[INFO] 有效結果筆數：3001
  [1/5]  set_id       : 192259368
           outfit title : Hanging With Your Bestie
           total_clo    : 0.7600
  items (6) :
  

<Figure size 760x440 with 1 Axes>

  [2/5]  set_id       : 220721299
           outfit title : Raw Edge Denim
           total_clo    : 1.1500
  items (5) :
         200527064  valentino rockstud rolling denim shoulder
         204861593  Anouki Denim Liana Embellished Brogues
         202067101  sky blue raw edge denim
         191494119  fendi lei cat-eye gold-tone sunglasse
         195792340  roberto cavalli embroidered denim dress
  最接近參考值 : 1.14  (source=ensemble, label={'description': 'Trousers, long-sleeve shirt plus suit jacket, vest, T-shirt', 'clo': 1.14})


<Figure size 760x440 with 1 Axes>

  [3/5]  set_id       : 208264087
           outfit title : Vegan Leather Boots
           total_clo    : 0.6600
  items (5) :
         184947151  braided handmade black leather shoulder
         182579270  FC Select Design New Vegan Meredith Patent Ankle Boots
         191683321  etro heart locket necklace
         183391603  COLLARED T-NECK LONG SLEEVED BLOUSE
         182055692  Women's Topshop Embroidered Ripped Mom Jeans
  最接近參考值 : 0.66  (source=ensemble, label={'description': 'Sports long sleeve 2 + Sweatpants (thick)', 'clo': 0.66})


...

  [4/5]  set_id       : 206357745
           outfit title : Bold Accessories
           total_clo    : 0.8900
  items (4) :
         148460893  lady lux beach handbag accessory
         170741790  Minna Laceup Leather Thong Sandals
         153111620  lady lux hand chain bohemian
         148363367  stripe it rich two piece
  最接近參考值 : 0.89  (source=ensemble, label={'description': 'Overalls, long-sleeve shirt, T-shirt', 'clo': 0.89})


<Figure size 760x250 with 1 Axes>

  [5/5]  set_id       : 224657449
           outfit title : Espadrilles
           total_clo    : 1.0000
  items (4) :
         213279889  sequin pineapple tassel straw bag
         179158981  valentino rockstud stripe leather espadrille
         202722132  banana republic elizabeth cole pineapple
         146587707  lucluc green color block scoop
  最接近參考值 : 1.0  (source=ensemble, label={'description': 'Bikini style briefs + Non-wired bra + Sleeveless blouse + Pencil skirt + Tights (10 denier) + Shoes', 'clo': 1.0})


<Figure size 760x250 with 1 Axes>

[完成] 小樣本抽查結束


# 分布穩定性

In [9]:
# ============================================================
# Cell: CLO 分布穩定性摘要（train + test + valid，本地 WSL 版）
# ============================================================
import os, json, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ================================================================
# 路徑設定
# ================================================================
BASE_DIR = "/home/ester/valid_description"

RESULT_FILES = {
    "train": os.path.join(BASE_DIR, "generate_CLO", "train_clo_results.jsonl"),
    "test" : os.path.join(BASE_DIR, "generate_CLO", "test_clo_results.jsonl"),
    "valid": os.path.join(BASE_DIR, "generate_CLO", "valid_clo_results.jsonl"),
}
BIN_NUM = 40

# ================================================================
# 讀取 total_clo（三檔合併）
# ================================================================
clo_values  = []
split_stats = {}   # 記錄每個 split 各自讀了幾筆，方便檢查

print("=== 讀取結果檔 ===")
for split, path in RESULT_FILES.items():
    if not os.path.exists(path):
        print(f"  ❌  {split}: 找不到 {path}")
        split_stats[split] = 0
        continue
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rec = json.loads(line)
                v   = rec.get("total_clo")
                if isinstance(v, (int, float)) and math.isfinite(v):
                    clo_values.append(float(v))
                    count += 1
            except: pass
    split_stats[split] = count
    print(f"  ✅  {split}: {count} 筆  ({path})")

print(f"\n  合計有效筆數：{len(clo_values)}\n")

# ================================================================
# 統計量
# ================================================================
if not clo_values:
    print("[ERROR] 沒有有效的 total_clo 資料，請確認路徑")
else:
    arr       = np.array(clo_values)
    q1        = float(np.percentile(arr, 25))
    median    = float(np.median(arr))
    q3        = float(np.percentile(arr, 75))
    iqr       = q3 - q1
    iqr_lower = q1 - 1.5 * iqr
    iqr_upper = q3 + 1.5 * iqr
    n_outliers = int(np.sum((arr < iqr_lower) | (arr > iqr_upper)))

    p99    = float(np.percentile(arr, 99))
    x_zoom = min(p99, iqr_upper * 2.5)
    x_zoom = max(x_zoom, q3 + iqr * 2)

    stats = {
        "count":           len(arr),
        "mean":            float(np.mean(arr)),
        "std":             float(np.std(arr, ddof=1)),
        "min":             float(np.min(arr)),
        "q1":              q1,
        "median":          median,
        "q3":              q3,
        "max":             float(np.max(arr)),
        "iqr":             iqr,
        "iqr_lower_bound": iqr_lower,
        "iqr_upper_bound": iqr_upper,
        "n_outliers":      n_outliers,
    }

    # ================================================================
    # 摘要表
    # ================================================================
    int_keys = {"count", "n_outliers"}
    print("=" * 44)
    print("   CLO 分布穩定性摘要（train + test + valid）")
    print("=" * 44)
    for k, v in stats.items():
        val_str = f"{v:.0f}" if k in int_keys else f"{v:.4f}"
        print(f"   {k.ljust(18)}  {val_str}")
    print("=" * 44)
    print(f"   IQR 界外共 {n_outliers} 筆（{100*n_outliers/len(arr):.1f}%）")
    print("=" * 44)

    # ================================================================
    # 配色
    # ================================================================
    C_MAIN   = "#4C72B0"
    C_MEAN   = "#DD8452"
    C_MEDIAN = "#55A868"
    C_BOUND  = "#C44E52"

    # split 顏色（給縮圖用）
    SPLIT_COLORS = {"train": "#4C72B0", "test": "#55A868", "valid": "#DD8452"}

    # ================================================================
    # 圖面配置：左=主直方圖, 右上=全距縮圖, 右下=箱型圖
    # ================================================================
    fig = plt.figure(figsize=(15, 6))
    gs  = gridspec.GridSpec(2, 2, figure=fig,
                            width_ratios=[1.6, 1],
                            height_ratios=[1, 1.6],
                            hspace=0.45, wspace=0.35)

    ax_hist = fig.add_subplot(gs[:, 0])
    ax_over = fig.add_subplot(gs[0, 1])
    ax_box  = fig.add_subplot(gs[1, 1])

    # ----------------------------------------------------------------
    # (A) 主直方圖（zoom）
    # ----------------------------------------------------------------
    clipped   = np.clip(arr, 0, x_zoom)
    n_clipped = int(np.sum(arr > x_zoom))

    ax_hist.hist(clipped, bins=BIN_NUM, range=(float(arr.min()), x_zoom),
                 color=C_MAIN, edgecolor="white", linewidth=0.4, alpha=0.85)
    ax_hist.axvline(stats["mean"],   color=C_MEAN,   lw=1.8, ls="--",
                    label=f"mean = {stats['mean']:.3f}")
    ax_hist.axvline(stats["median"], color=C_MEDIAN, lw=1.8, ls=":",
                    label=f"median = {stats['median']:.3f}")
    ax_hist.axvline(iqr_lower, color=C_BOUND, lw=1.3, ls="-.",
                    label=f"IQR bounds [{iqr_lower:.2f}, {iqr_upper:.2f}]")
    ax_hist.axvline(iqr_upper, color=C_BOUND, lw=1.3, ls="-.")
    ax_hist.set_xlim(left=max(float(arr.min()) - 0.05, -0.1), right=x_zoom)
    ax_hist.set_title(
        f"CLO Histogram — train({split_stats.get('train',0)}) + "
        f"test({split_stats.get('test',0)}) + "
        f"valid({split_stats.get('valid',0)})",
        fontsize=10
    )
    ax_hist.set_xlabel("total_clo")
    ax_hist.set_ylabel("count")
    ax_hist.legend(fontsize=8, loc="upper right")
    ax_hist.grid(axis="y", alpha=0.3, lw=0.7)

    if n_clipped > 0:
        ax_hist.annotate(
            f"▶ {n_clipped} values\n  > {x_zoom:.1f} not shown",
            xy=(x_zoom, ax_hist.get_ylim()[1] * 0.95),
            ha="right", va="top", fontsize=7.5, color=C_BOUND,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C_BOUND, alpha=0.7)
        )

    # ----------------------------------------------------------------
    # (B) 右上：全距縮圖，三個 split 各自疊加不同顏色
    # ----------------------------------------------------------------
    # 重新讀一次，把三個 split 分開存
    split_values = {}
    for split, path in RESULT_FILES.items():
        vals = []
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        rec = json.loads(line)
                        v   = rec.get("total_clo")
                        if isinstance(v, (int, float)) and math.isfinite(v):
                            vals.append(float(v))
                    except: pass
        split_values[split] = np.array(vals) if vals else np.array([])

    bins_over = np.linspace(float(arr.min()), float(arr.max()), 80)
    for split, color in SPLIT_COLORS.items():
        sv = split_values.get(split, np.array([]))
        if len(sv) == 0: continue
        ax_over.hist(sv, bins=bins_over, color=color, edgecolor="none",
                     alpha=0.55, label=f"{split} ({len(sv)})")
    ax_over.axvline(iqr_upper, color=C_BOUND, lw=1.0, ls="-.")
    ax_over.set_title("Full range (by split)", fontsize=9)
    ax_over.set_xlabel("total_clo", fontsize=8)
    ax_over.set_ylabel("count",     fontsize=8)
    ax_over.tick_params(labelsize=7)
    ax_over.legend(fontsize=7, loc="upper right")
    ax_over.grid(axis="y", alpha=0.25, lw=0.6)

    # ----------------------------------------------------------------
    # (C) 箱型圖（zoom）
    # ----------------------------------------------------------------
    ax_box.boxplot(
        arr, vert=True, patch_artist=True, widths=0.45,
        boxprops    =dict(facecolor=C_MAIN, color="#2c4a7c", alpha=0.75),
        medianprops =dict(color=C_MEAN, linewidth=2.5),
        whiskerprops=dict(color="#2c4a7c", linewidth=1.2),
        capprops    =dict(color="#2c4a7c", linewidth=1.5),
        flierprops  =dict(marker="o", color=C_BOUND, alpha=0.30, markersize=2.5)
    )
    ax_box.set_ylim(bottom=max(float(arr.min()) - 0.1, -0.15), top=x_zoom)

    for lbl, val, x_txt, ha in [
        ("Q1",     q1,     -1.38, "right"),
        ("Median", median,  1.38, "left"),
        ("Q3",     q3,     -1.38, "right"),
    ]:
        ax_box.annotate(
            f"{lbl} = {val:.3f}",
            xy=(1, val), xytext=(x_txt, val),
            ha=ha, va="center", fontsize=8, color="#222222",
            arrowprops=dict(arrowstyle="-", color="#bbbbbb", lw=0.7)
        )

    n_above = int(np.sum(arr > x_zoom))
    if n_above > 0:
        ax_box.annotate(
            f"↑ {n_above} pts > {x_zoom:.1f}",
            xy=(1, x_zoom), xytext=(1.25, x_zoom * 0.92),
            fontsize=7.5, color=C_BOUND,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C_BOUND, alpha=0.7)
        )

    ax_box.set_title("CLO Box Plot (zoomed)", fontsize=10)
    ax_box.set_ylabel("total_clo", fontsize=9)
    ax_box.set_xticks([])
    ax_box.grid(axis="y", alpha=0.3, lw=0.7)

    # ================================================================
    # 總標題
    # ================================================================
    plt.suptitle(
        f"CLO Distribution Summary — train + test + valid  "
        f"(n={len(arr)},  outliers={n_outliers} / {100*n_outliers/len(arr):.1f}%)",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()

=== 讀取結果檔 ===
  ✅  train: 16995 筆  (/home/ester/valid_description/generate_CLO/train_clo_results.jsonl)
  ✅  test: 15145 筆  (/home/ester/valid_description/generate_CLO/test_clo_results.jsonl)
  ✅  valid: 3000 筆  (/home/ester/valid_description/generate_CLO/valid_clo_results.jsonl)

  合計有效筆數：35140

   CLO 分布穩定性摘要（train + test + valid）
   count               35140
   mean                1.0066
   std                 0.5586
   min                 0.0000
   q1                  0.6300
   median              0.9300
   q3                  1.2125
   max                 5.2000
   iqr                 0.5825
   iqr_lower_bound     -0.2437
   iqr_upper_bound     2.0862
   n_outliers          1726
   IQR 界外共 1726 筆（4.9%）


/tmp/ipykernel_18501/751891234.py:238: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


<Figure size 1500x600 with 3 Axes>